<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day17_practice1_%EC%96%B4%ED%85%90%EC%85%98_%EC%86%90%EA%B3%84%EC%82%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 어텐션 손계산 - 질문 · 색인 · 내용 (Q · K · V)
# 평균 풀링: 모든 단어를 '똑같은 무게'로 (1/n)씩
# 어텐션   : 단어마다 '관련도만큼의 무게'로
# Q (Query) : 내가 던진 질문
# K (key)   : 질문과 비교하는 각 단어
# V (Value) : 각 단어가 가진 원래 정보 내용
# 어텐션의 결과물 : 각 단어에 관련도를 곱한 전체 문장 내용

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)

In [ ]:
# 단어 3개 ('영화', '결말', '재미있다')의 Q, K, V
Q = torch.tensor(([1.0, 0.0], # '영화'를 질문으로
                  [0.0, 1.0], # '정말'을 질문으로
                  [2.0, 0.0])) # '재미있다'를 질문으로

K = torch.tensor(([1.0, 0.0], # 영화
                  [0.0, 1.0], # 정말
                  [1.0, 0.0])) # 재미있다

V = torch.tensor(([10., 0.], # 영화 의 내용
                  [0., 10.], # 정말 의 내용
                  [5., 5.])) # 재미있다 의 내용

In [ ]:
# 1. 질문과 모든 단어를 내적 (관련도 점수)
q = Q[2] # '재미있다'를 질문으로
scores = torch.tensor([q @ K[0], q @ K[1], q @ K[2]]) # q와 각 K의 단어들을 내적(행렬곱) → 얼마나 관련 있나 점수
print(" 관련도 점수 (내적):", scores.tolist())

 관련도 점수 (내적): [2.0, 0.0, 2.0]


In [ ]:
# 2. √d 로 나누기 (d=2) — 점수 스케일 안정화
import math
scaled = scores / math.sqrt(2)
print(" 스케일 조정 (÷√2):", [round(v, 3) for v in scaled.tolist()])

 스케일 조정 (÷√2): [1.414, 0.0, 1.414]


In [ ]:
# 3. softmax - 점수를 '무게 합=1'로
weights = F.softmax(scaled, dim=0) # 1차원 축 기준
print(" 어텐션 가중치 (softmax):", [round(v, 3) for v in weights.tolist()])

 어텐션 가중치 (softmax): [0.446, 0.108, 0.446]


In [ ]:
# 4. 내용(V)의 가중 평균
out_hand = weights[0]*V[0] + weights[1]*V[1] + weights[2]*V[2]
print(" 가중합 결과(어텐션 결과):", [round(v, 3) for v in out_hand.tolist()])


 가중합 결과(어텐션 결과): [6.687, 3.313]


In [ ]:
# 검증
attn = F.softmax(Q @ K.T / math.sqrt(2), dim=-1) #dim=-1 ← 마지막 축 (각 행 기준)
out_all = attn @ V # 무게(가중치) x 내용
print("\n어텐션 맵 (행=질문 단어, 열=참조 단어):")
print("           영화      정말   재미있다")
for i, name in enumerate({"영화   ", "정말    ", "재미있다"}):
  print(f" {name}: {[round(v, 3) for v in attn[i].tolist()]}")

  assert torch.allclose(out_all[2], out_hand, atol=1e-6)


어텐션 맵 (행=질문 단어, 열=참조 단어):
           영화      정말   재미있다
 영화   : [0.401, 0.198, 0.401]
 재미있다: [0.248, 0.503, 0.248]
 정말    : [0.446, 0.108, 0.446]
